In [8]:
%pip install pandas requests mesa numpy pyarrow seaborn matplotlib


  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached matplotlib-3.10.9-cp311-cp311-win_amd64.whl.metadata (52 kB)
  Using cached contourpy-1.3.3-cp311-cp311-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached kiwisolver-1.5.0-cp311-cp311-win_amd64.whl.metadata (5.2 kB)
  Using cached pillow-12.2.0-cp311-cp311-win_amd64.whl.metadata (9.0 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
Using cached matplotlib-3.10.9-cp311-cp311-win_amd64.whl (8.2 MB)
Using cached contourpy-1.3.3-cp311-cp311-win_amd64.whl (225 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 2.4/2.4 MB 14.9 MB/s  0:00:00
Using cached kiwisolver-1.5.0-cp311-cp311-win_amd64.whl (73 kB)
Using cached pillow-12.2.0-cp311-cp311-win_amd64.whl (7.1 MB)

In [5]:
import requests
import mesa
import pandas as pd
import random
from datetime import datetime, timedelta
import time


In [9]:

# --- EVENT LOGGER ---
class ProcessLogger:
    def __init__(self):
        self.log = []
        self.step_counter = 0 # To track event order within a single step

    def add_event(self, step, case_id, activity, timestamp, **kwargs):
        self.step_counter += 1
        event = {
            "Step": step, 
            "StepCounter": self.step_counter, 
            "CaseID": case_id, 
            "Activity": activity, 
            "Timestamp": timestamp
        }
        event.update(kwargs)
        self.log.append(event)

    def to_dataframe(self):
        return pd.DataFrame(self.log)

# --- MINER AGENT ---
class MinerAgent(mesa.Agent):
    def __init__(self, model, name, hash_power, pwr_kw, elec_price, miner_type):
        super().__init__(model) 
        self.name = name        
        self.hash_power = hash_power       
        self.pwr_kw = pwr_kw               
        self.elec_price = elec_price       
        self.miner_type = miner_type       
        self.is_active = False
        self.btc_balance = 0.0
        self.profit = 0.0

    def evaluate_participation(self, current_time, expected_reward_value):
        # Unique CaseID per miner per round prevents "spaghetti" process maps
        case_id = f"{self.name}_Round_{self.model.steps}"

        # Expected cost is calculated using the expected block time of ten minutes.
        expected_cost = self.pwr_kw * (10.0 / 60.0) * self.elec_price
        
        # Expected revenue is calculated from the probability of winning multiplied by expected block reward.
        total_hash_rate = sum(a.hash_power for a in self.model.agents)
        probability_of_winning = self.hash_power / total_hash_rate
        expected_revenue = probability_of_winning * expected_reward_value
        
        expected_profit = expected_revenue - expected_cost
        
        # Log: evaluating profitability 
        self.model.logger.add_event(self.model.steps, case_id, "Evaluate_Profitability", current_time, Expected_Profit=expected_profit)
        
        # A miner participates only when expected profit is positive.
        if expected_profit > 0:
            self.is_active = True
            # Log: activating mining 
            self.model.logger.add_event(self.model.steps, case_id, "Activate_Mining", current_time)
        else:
            self.is_active = False
            # Log: skipping mining
            self.model.logger.add_event(self.model.steps, case_id, "Skip_Mining", current_time)

# --- BLOCKCHAIN MODEL ---
class BitcoinMiningModel(mesa.Model):
    def __init__(self):
        super().__init__() 
        self.logger = ProcessLogger()
        self.steps = 0 
        self.current_time = datetime(2026, 1, 1, 0, 0, 0)
        self.btc_price = 60000 
        self.block_reward = 3.125
        
        # Realistic hardware abstractions: Name, Hash Rate, Power (kW), Elec Price ($/kWh), Type 
        pool_configs = [
            ("Pool_A_Giant", 150.0, 4000000, 0.04, "Mining Pool"),
            ("Pool_B_Large", 100.0, 2800000, 0.05, "Mining Pool"),
            ("Miner_C_Industrial", 20.0, 600000, 0.03, "Industrial Miner"),
            ("Miner_D_Medium", 5.0, 160000, 0.06, "Medium-Scale Miner")
        ]
        
        for name, hash_pwr, pwr_kw, elec_price, miner_type in pool_configs:
            # Mesa 3.0+ automatically adds initialized agents to self.agents
            MinerAgent(self, name, hash_pwr, pwr_kw, elec_price, miner_type)

    def step(self):
        self.steps += 1 
        self.btc_price += random.uniform(-1500, 1500) # Market price fluctuation 
        expected_reward_value = self.block_reward * self.btc_price
        
        # Phase 1: Pre-round Evaluation
        for agent in self.agents:
            agent.evaluate_participation(self.current_time, expected_reward_value)
            
        active_agents = [a for a in self.agents if a.is_active]
        
        # If no agents are active, skip ahead 10 minutes (prevents errors)
        if not active_agents:
            self.current_time += timedelta(minutes=10)
            return

        # Phase 2: Block Discovery
        # The duration of each block-mining cycle is sampled from an exponential distribution with a mean of ten minutes.
        block_time_mins = random.expovariate(1.0 / 10.0) 
        self.current_time += timedelta(minutes=block_time_mins)
        
        total_active_hash = sum(a.hash_power for a in active_agents)
        weights = [a.hash_power / total_active_hash for a in active_agents]
        
        # The probability that miner (i) wins the block is proportional to its share of the total active hash rate.
        winner = random.choices(active_agents, weights=weights, k=1)[0]
        
        # Phase 3: Cost Accounting and Process Logging
        for agent in active_agents:
            # Maintain the unique CaseID matching Phase 1
            case_id = f"{agent.name}_Round_{self.steps}"
            
            # Log: starting mining 
            self.logger.add_event(self.steps, case_id, "Start_Mining", self.current_time)
            
            # Realised cost is calculated based on sampled block time.
            realized_cost = agent.pwr_kw * (block_time_mins / 60.0) * agent.elec_price
            agent.profit -= realized_cost
            
            # Log: consuming electricity 
            self.logger.add_event(self.steps, case_id, "Consume_Electricity", self.current_time, Cost=realized_cost)
            
            if agent == winner:
                agent.btc_balance += self.block_reward
                realized_revenue = self.block_reward * self.btc_price
                agent.profit += realized_revenue
                
                # Log: winning a block, receiving rewards 
                self.logger.add_event(self.steps, case_id, "Win_Block", self.current_time)
                self.logger.add_event(self.steps, case_id, "Receive_Reward", self.current_time, Revenue=realized_revenue)
            else:
                # Log: losing a block 
                self.logger.add_event(self.steps, case_id, "Lose_Block", self.current_time)
                
            # Log: updating profit 
            self.logger.add_event(self.steps, case_id, "Update_Profit", self.current_time, Net_Profit=agent.profit)

# --- RUN SIMULATION ---
if __name__ == "__main__":
    model = BitcoinMiningModel()

    print("Simulating 100 block-mining cycles...")
    for i in range(100): 
        model.step()

    event_log_df = model.logger.to_dataframe()
    print(f"\nSimulation complete! Generated {len(event_log_df)} events.")
    print(event_log_df.head(15))

    # Ready for export
    event_log_df.to_csv("bitcoin_agent_event_log.csv", index=False)

Simulating 100 block-mining cycles...

Simulation complete! Generated 2500 events.
    Step  StepCounter                      CaseID                Activity  \
0      2            1        Pool_A_Giant_Round_2  Evaluate_Profitability   
1      2            2        Pool_A_Giant_Round_2         Activate_Mining   
2      2            3        Pool_B_Large_Round_2  Evaluate_Profitability   
3      2            4        Pool_B_Large_Round_2         Activate_Mining   
4      2            5  Miner_C_Industrial_Round_2  Evaluate_Profitability   
5      2            6  Miner_C_Industrial_Round_2         Activate_Mining   
6      2            7      Miner_D_Medium_Round_2  Evaluate_Profitability   
7      2            8      Miner_D_Medium_Round_2         Activate_Mining   
8      2            9        Pool_A_Giant_Round_2            Start_Mining   
9      2           10        Pool_A_Giant_Round_2     Consume_Electricity   
10     2           11        Pool_A_Giant_Round_2               Win_Bl